# SPY Risk Alert Project Pipeline

This cumulative pipeline covers Stage04 ingestion, Stage05 storage, and Stage06 preprocessing. It defaults to retained raw data; set `REFRESH_RAW = True` only for an intentional new Nasdaq acquisition. The source is unadjusted OHLCV, and no return target or trading recommendation is created here.

## 1. Project Root, Configuration, and Imports

In [1]:
# --- run me first ---
from pathlib import Path
import os, sys

if Path.cwd().name == 'notebooks':
    os.chdir('..')  # project/notebooks -> project
ROOT = Path.cwd()
if not (ROOT / 'src' / 'ingestion.py').is_file():
    for candidate in (ROOT, *ROOT.parents):
        project_candidate = candidate / 'project'
        if (project_candidate / 'src' / 'ingestion.py').is_file():
            ROOT = project_candidate
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datetime import date

from src.cleaning import clean_spy_ohlcv, write_cleaning_report
from src.config import get_processed_data_dir, get_raw_data_dir, load_env
from src.ingestion import (
    fetch_nasdaq_history, timestamp_utc, validate_spy_history, write_manifest, write_raw_csv
)
from src.storage import get_parquet_engine, read_df, validate_roundtrip, write_df

environment_loaded = load_env()
RAW_DIR = get_raw_data_dir()
PROCESSED_DIR = get_processed_data_dir()
print('working from:', ROOT.name)
print('Environment loaded:', environment_loaded)
print('Raw directory:', RAW_DIR)
print('Processed directory:', PROCESSED_DIR)

working from: project
Environment loaded: True
Raw directory: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project/data/raw
Processed directory: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project/data/processed


## 2. Stage04 Raw Snapshot

Normal runs reuse the latest retained raw CSV. An explicit refresh validates and saves a new raw snapshot plus manifest.

In [2]:
REFRESH_RAW = False
SYMBOL = 'SPY'
csv_schema = {
    'open': 'float64', 'high': 'float64', 'low': 'float64',
    'close': 'float64', 'volume': 'int64',
}
raw_candidates = sorted(RAW_DIR.glob('api_nasdaq_spy_daily_*.csv'))

if REFRESH_RAW:
    end_date = date.today()
    start_date = end_date.replace(year=end_date.year - 10)
    spy_raw, source_metadata = fetch_nasdaq_history(
        SYMBOL, start_date=start_date.isoformat(), end_date=end_date.isoformat()
    )
    validation = validate_spy_history(spy_raw)
    snapshot_timestamp = timestamp_utc()
    raw_path = write_raw_csv(
        spy_raw, RAW_DIR, 'api_nasdaq_spy_daily', timestamp=snapshot_timestamp
    )
    manifest_path = write_manifest(
        {
            'path': raw_path, 'dataset': 'SPY daily unadjusted OHLCV',
            'rows': len(spy_raw), 'columns': list(spy_raw.columns),
            'source_metadata': source_metadata, 'validation': validation,
        },
        RAW_DIR / f'ingestion_manifest_{snapshot_timestamp}.json',
    )
    print('Refreshed raw snapshot:', raw_path.name)
    print('Saved manifest:', manifest_path.name)
else:
    if not raw_candidates:
        raise FileNotFoundError('No Stage04 SPY raw snapshot found. Set REFRESH_RAW = True.')
    raw_path = raw_candidates[-1]
    snapshot_timestamp = raw_path.stem.removeprefix('api_nasdaq_spy_daily_')
    spy_raw = read_df(raw_path, parse_dates=['date'], dtype=csv_schema)
    validation = validate_spy_history(spy_raw)
    print('Reused raw snapshot:', raw_path.name)

print('Rows and columns:', validation['shape'])
print('Date range:', validation['date_min'], 'to', validation['date_max'])
spy_raw.head()

Reused raw snapshot: api_nasdaq_spy_daily_20260907-143336.csv
Rows and columns: [2512, 6]
Date range: 2016-09-07 to 2026-09-04


,date,open,high,low,close,volume
0,2016-09-07,218.84,219.2200,218.30,219.01,76302150
1,2016-09-08,218.62,218.9400,218.15,218.51,73855230
2,2016-09-09,216.97,217.0300,213.25,213.28,220309300
3,2016-09-12,212.39,216.8100,212.31,216.34,167653400
4,2016-09-13,214.84,215.1499,212.50,213.23,182323200


## 3. Stage05 Typed Storage

The Parquet file is a typed representation of the named raw snapshot, not a cleaning or feature step.

In [3]:
storage_path = PROCESSED_DIR / f'spy_ohlcv_nasdaq_{snapshot_timestamp}.parquet'
write_df(spy_raw, storage_path)
spy_stored = read_df(storage_path)
storage_roundtrip = validate_roundtrip(
    spy_raw, spy_stored,
    {'date': 'datetime', 'open': 'float', 'close': 'float', 'volume': 'integer'},
)
if not storage_roundtrip['passed']:
    raise ValueError(f'Stage05 storage validation failed: {storage_roundtrip}')
print('Parquet engine:', get_parquet_engine())
print('Stored file:', storage_path.name)
print('Storage round-trip passed:', storage_roundtrip['passed'])

Parquet engine: pyarrow
Stored file: spy_ohlcv_nasdaq_20260907-143336.parquet
Storage round-trip passed: True


## 4. Stage06 Deterministic Preprocessing

The policy reparses and validates canonical OHLCV fields, sorts dates, and records its non-actions. It does not impute prices, remove outliers, or fit a global scaler.

In [4]:
spy_clean, cleaning_report = clean_spy_ohlcv(spy_stored)
preprocessed_path = PROCESSED_DIR / f'spy_ohlcv_preprocessed_{snapshot_timestamp}.parquet'
report_path = PROCESSED_DIR / f'cleaning_report_{snapshot_timestamp}.json'
write_df(spy_clean, preprocessed_path)
write_cleaning_report(cleaning_report, report_path)
spy_reloaded = read_df(preprocessed_path)
preprocessing_roundtrip = validate_roundtrip(
    spy_clean, spy_reloaded,
    {'date': 'datetime', 'open': 'float', 'close': 'float', 'volume': 'integer'},
)
if not preprocessing_roundtrip['passed']:
    raise ValueError(f'Stage06 preprocessing validation failed: {preprocessing_roundtrip}')

print('Preprocessed file:', preprocessed_path.name)
print('Cleaning report:', report_path.name)
print('Rows dropped:', cleaning_report['rows_dropped'])
print('Preprocessing round-trip passed:', preprocessing_roundtrip['passed'])

Preprocessed file: spy_ohlcv_preprocessed_20260907-143336.parquet
Cleaning report: cleaning_report_20260907-143336.json
Rows dropped: 0
Preprocessing round-trip passed: True


## Sources, Storage, Preprocessing, Assumptions, and Risks

- Source rules: `docs/data_sources.md`; storage lineage: `docs/data_storage.md`; preprocessing policy: `docs/preprocessing.md`.
- `data/raw/` remains immutable; processed outputs are reproducible from the named raw snapshot and code.
- The source remains unadjusted OHLCV. Corporate-action handling, return definition, outlier treatment, and training-only scaling remain future decisions.
- Later stages extend this same notebook with risk analysis, EDA, features, modeling, and reporting.